# Pixel 2022 Code

In [1]:
import rasterio
import rasterio.plot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import geopandas as gpd

In [ ]:

# Set your shapefile and raster directories
raster_dir = r'D:\Capstone\planet_image\Plot_map_2022(correct)_planet_CC0%_psscene_analytic_sr_udm2 (1)\PSScene'


# List all raster files in the directory (e.g., .tif)
raster_files_2022 = [os.path.join(raster_dir, f) for f in os.listdir(raster_dir) if f.endswith('SR_clip.tif')]

print("Loaded raster files:", raster_files_2022)

Loaded raster files: ['D:\\Capstone\\planet_image\\Plot_map_2022(correct)_planet_CC0%_psscene_analytic_sr_udm2 (1)\\PSScene\\20220508_153525_43_24a5_3B_AnalyticMS_SR_clip.tif', 'D:\\Capstone\\planet_image\\Plot_map_2022(correct)_planet_CC0%_psscene_analytic_sr_udm2 (1)\\PSScene\\20220508_153527_74_24a5_3B_AnalyticMS_SR_clip.tif', 'D:\\Capstone\\planet_image\\Plot_map_2022(correct)_planet_CC0%_psscene_analytic_sr_udm2 (1)\\PSScene\\20220509_154511_72_227a_3B_AnalyticMS_SR_clip.tif', 'D:\\Capstone\\planet_image\\Plot_map_2022(correct)_planet_CC0%_psscene_analytic_sr_udm2 (1)\\PSScene\\20220510_150455_04_2465_3B_AnalyticMS_SR_clip.tif', 'D:\\Capstone\\planet_image\\Plot_map_2022(correct)_planet_CC0%_psscene_analytic_sr_udm2 (1)\\PSScene\\20220510_154526_12_2424_3B_AnalyticMS_SR_clip.tif', 'D:\\Capstone\\planet_image\\Plot_map_2022(correct)_planet_CC0%_psscene_analytic_sr_udm2 (1)\\PSScene\\20220512_153350_32_2492_3B_AnalyticMS_SR_clip.tif', 'D:\\Capstone\\planet_image\\Plot_map_2022(corre

In [4]:
pixel_2022_32618 = gpd.read_file(r'D:\Capstone\2022\pixel_2022_32618.shp')


In [5]:
print(pixel_2022_32618.crs)


EPSG:32618


In [7]:
pixel_2022_32618

,id,pixel_numb,pixel_iden,N_rate,plant_date,geometry
0,1,1,NaN,100.0,2023-05-16,"POLYGON ((380010.000 4700601.000, 380013.000 4..."
1,2,2,NaN,100.0,2023-05-16,"POLYGON ((380043.000 4700613.000, 380046.000 4..."
2,3,3,NaN,0.0,2023-05-16,"POLYGON ((380067.000 4700622.000, 380070.000 4..."
3,4,4,NaN,100.0,2023-05-16,"POLYGON ((380124.000 4700646.000, 380127.000 4..."
4,5,5,NaN,50.0,2023-05-16,"POLYGON ((380010.000 4700592.000, 380013.000 4..."
5,6,6,NaN,0.0,2023-05-16,"POLYGON ((380034.000 4700601.000, 380037.000 4..."
6,7,7,NaN,50.0,2023-05-16,"POLYGON ((380073.000 4700616.000, 380076.000 4..."
7,8,8,NaN,0.0,2023-05-16,"POLYGON ((380103.000 4700628.000, 380106.000 4..."
8,9,9,NaN,50.0,2023-05-16,"POLYGON ((380133.000 4700640.000, 380136.000 4..."
9,10,10,NaN,100.0,2023-06-02,"POLYGON ((380016.000 4700583.000, 380019.000 4..."


In [25]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import os

def clip_save_and_store_rasters(raster_path, polygons_gdf, base_output_dir):
    """
    Clips a raster using polygons, saves each segment to disk,
    and returns the data in a list.
    """
    clipped_pixels_list = []
    try:
        with rasterio.open(raster_path) as src:
            # Reproject polygons if their CRS doesn't match the raster's
            polygons_to_use = polygons_gdf
            if polygons_gdf.crs != src.crs:
                print(f"  - CRS mismatch. Reprojecting polygons for {os.path.basename(raster_path)}.")
                polygons_to_use = polygons_gdf.to_crs(src.crs)

            image_name = os.path.splitext(os.path.basename(raster_path))[0]

            for index, row in polygons_to_use.iterrows():
                pixel_nb = row['pixel_numb']
                output_name = f"{image_name}_pixel_{pixel_nb}"
                
                try:
                    clipped_array, clipped_transform = mask(dataset=src, shapes=[row['geometry']], crop=True)
                except ValueError:
                    # This happens if a polygon doesn't overlap the raster, skip it
                    continue

                # --- 1. SAVE TO DISK LOGIC ---
                output_folder = os.path.join(base_output_dir, f"pixel_{pixel_nb}")
                os.makedirs(output_folder, exist_ok=True)
                output_filepath = os.path.join(output_folder, f"{output_name}.tif")

                out_meta = src.meta.copy()
                out_meta.update({
                    "driver": "GTiff",
                    "height": clipped_array.shape[1],
                    "width": clipped_array.shape[2],
                    "transform": clipped_transform,
                    "crs": src.crs
                })

                with rasterio.open(output_filepath, "w", **out_meta) as dest:
                    dest.write(clipped_array)

                # --- 2. STORE IN LIST LOGIC ---
                clipped_pixels_list.append({
                    'name': output_name,
                    'pixels': clipped_array
                })
    except Exception as e:
        print(f"Could not process {raster_path}. Error: {e}")
        return []

    return clipped_pixels_list

# --- 1. DEFINE YOUR MAIN OUTPUT FOLDER ---
output_dir = r"D:\Capstone\Data\2022\pixel_2022_raster"
os.makedirs(output_dir, exist_ok=True)

# --- (Your existing code to define raster_files_2021 and pixel_2021_32618) ---
# raster_dir = 'path/to/your/raster_files'
# shapefile_path = 'path/to/your/field_polygons.shp'
# raster_files_2021 = [os.path.join(raster_dir, f) for f in os.listdir(raster_dir) if f.endswith('SR_clip.tif')]
# pixel_2021_32618 = gpd.read_file(shapefile_path)
# ----------------------------------------------------------------------------

# --- 3. Process Each Raster in the List ---
# This list will hold ALL the clipped data from ALL rasters
all_clipped_data = []

print(f"\nFound {len(raster_files_2022)} rasters to process.")
print(f"Output files will be saved in: {os.path.abspath(output_dir)}")

# Loop through each raster file path
for raster_path in raster_files_2022:
    print(f"\n--- Processing raster: {os.path.basename(raster_path)} ---")
    
    # Call the function for the current raster, passing the output directory
    results_for_one_raster = clip_save_and_store_rasters(raster_path, pixel_2022_32618, output_dir)
    
    # Add the results from this raster to our main list
    if results_for_one_raster:
        all_clipped_data.extend(results_for_one_raster)
        print(f"  > Finished. Saved and stored {len(results_for_one_raster)} segments.")

# --- 4. Display Final Results ---
if all_clipped_data:
    print("\n\n==============================")
    print("All tasks complete!")
    print(f"Total files saved to disk: {len(all_clipped_data)}")
    print(f"Total items stored in all_clipped_data list: {len(all_clipped_data)}")
    
    # Example: Accessing the first clipped result from the combined list
    first_result = all_clipped_data[0]
    print(f"\nExample - Name of first clip: {first_result['name']}")
    print(f"Example - Shape of first pixel array: {first_result['pixels'].shape}")
else:
    print("\nNo data was clipped or stored.")


Found 33 rasters to process.
Output files will be saved in: D:\Capstone\Data\2022\pixel_2022_raster

--- Processing raster: 20220508_153525_43_24a5_3B_AnalyticMS_SR_clip.tif ---
  > Finished. Saved and stored 20 segments.

--- Processing raster: 20220508_153527_74_24a5_3B_AnalyticMS_SR_clip.tif ---
  > Finished. Saved and stored 20 segments.

--- Processing raster: 20220509_154511_72_227a_3B_AnalyticMS_SR_clip.tif ---
  > Finished. Saved and stored 20 segments.

--- Processing raster: 20220510_150455_04_2465_3B_AnalyticMS_SR_clip.tif ---
  > Finished. Saved and stored 20 segments.

--- Processing raster: 20220510_154526_12_2424_3B_AnalyticMS_SR_clip.tif ---
  > Finished. Saved and stored 20 segments.

--- Processing raster: 20220512_153350_32_2492_3B_AnalyticMS_SR_clip.tif ---
  > Finished. Saved and stored 20 segments.

--- Processing raster: 20220512_153352_62_2492_3B_AnalyticMS_SR_clip.tif ---
  > Finished. Saved and stored 20 segments.

--- Processing raster: 20220518_154328_11_24

In [26]:
all_clipped_data

[{'name': '20220508_153525_43_24a5_3B_AnalyticMS_SR_clip_pixel_1',
  'pixels': array([[[ 384,    0]],
  
         [[ 749,    0]],
  
         [[ 639,    0]],
  
         [[3910,    0]]], dtype=uint16)},
 {'name': '20220508_153525_43_24a5_3B_AnalyticMS_SR_clip_pixel_2',
  'pixels': array([[[ 369,    0],
          [   0,    0]],
  
         [[ 765,    0],
          [   0,    0]],
  
         [[ 633,    0],
          [   0,    0]],
  
         [[4157,    0],
          [   0,    0]]], dtype=uint16)},
 {'name': '20220508_153525_43_24a5_3B_AnalyticMS_SR_clip_pixel_3',
  'pixels': array([[[   0,    0],
          [ 459,    0],
          [   0,    0]],
  
         [[   0,    0],
          [ 821,    0],
          [   0,    0]],
  
         [[   0,    0],
          [ 789,    0],
          [   0,    0]],
  
         [[   0,    0],
          [3761,    0],
          [   0,    0]]], dtype=uint16)},
 {'name': '20220508_153525_43_24a5_3B_AnalyticMS_SR_clip_pixel_4',
  'pixels': array([[[ 384,    0],
  

In [30]:
import pandas as pd

# Define band names in order
band_names = ['blue', 'green', 'red', 'NIR']

pixel_records = []
for item in all_clipped_data:
    name = item['name']  # e.g., 'rastername_pixel_1'
    pixel_id = name.split('_')[-1]  # e.g., '1'
    arr = item['pixels']  # shape: (bands, height, width)
    # Flatten each band and create a dict for this pixel
    row = {'name': name}
    for i, band in enumerate(band_names):
        # Flatten the band array and store as a list
        row[f'pixel_{pixel_id}_{band}'] = arr[i].flatten().tolist()
    pixel_records.append(row)

df_pixel_2022= pd.DataFrame(pixel_records)
df_pixel_2022.head()

,name,pixel_1_blue,pixel_1_green,pixel_1_red,pixel_1_NIR,pixel_2_blue,pixel_2_green,pixel_2_red,pixel_2_NIR,pixel_3_blue,...,pixel_18_red,pixel_18_NIR,pixel_19_blue,pixel_19_green,pixel_19_red,pixel_19_NIR,pixel_20_blue,pixel_20_green,pixel_20_red,pixel_20_NIR
0,20220508_153525_43_24a5_3B_AnalyticMS_SR_clip_...,"[384, 0]","[749, 0]","[639, 0]","[3910, 0]",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20220508_153525_43_24a5_3B_AnalyticMS_SR_clip_...,NaN,NaN,NaN,NaN,"[369, 0, 0, 0]","[765, 0, 0, 0]","[633, 0, 0, 0]","[4157, 0, 0, 0]",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20220508_153525_43_24a5_3B_AnalyticMS_SR_clip_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[0, 0, 459, 0, 0, 0]",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20220508_153525_43_24a5_3B_AnalyticMS_SR_clip_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20220508_153525_43_24a5_3B_AnalyticMS_SR_clip_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
df_pixel_2022.to_csv(r'D:\Capstone\Data\2022\pixel_2022_data_not_sorted.csv', index=False)

In [32]:
df_pixel_2022.columns

Index(['name', 'pixel_1_blue', 'pixel_1_green', 'pixel_1_red', 'pixel_1_NIR',
       'pixel_2_blue', 'pixel_2_green', 'pixel_2_red', 'pixel_2_NIR',
       'pixel_3_blue', 'pixel_3_green', 'pixel_3_red', 'pixel_3_NIR',
       'pixel_4_blue', 'pixel_4_green', 'pixel_4_red', 'pixel_4_NIR',
       'pixel_5_blue', 'pixel_5_green', 'pixel_5_red', 'pixel_5_NIR',
       'pixel_6_blue', 'pixel_6_green', 'pixel_6_red', 'pixel_6_NIR',
       'pixel_7_blue', 'pixel_7_green', 'pixel_7_red', 'pixel_7_NIR',
       'pixel_8_blue', 'pixel_8_green', 'pixel_8_red', 'pixel_8_NIR',
       'pixel_9_blue', 'pixel_9_green', 'pixel_9_red', 'pixel_9_NIR',
       'pixel_10_blue', 'pixel_10_green', 'pixel_10_red', 'pixel_10_NIR',
       'pixel_11_blue', 'pixel_11_green', 'pixel_11_red', 'pixel_11_NIR',
       'pixel_12_blue', 'pixel_12_green', 'pixel_12_red', 'pixel_12_NIR',
       'pixel_13_blue', 'pixel_13_green', 'pixel_13_red', 'pixel_13_NIR',
       'pixel_14_blue', 'pixel_14_green', 'pixel_14_red', 'pixel_1

In [34]:

# List of columns you want to fix
columns_to_fix = ['pixel_1_blue', 'pixel_1_green', 'pixel_1_red', 'pixel_1_NIR',
       'pixel_2_blue', 'pixel_2_green', 'pixel_2_red', 'pixel_2_NIR',
       'pixel_3_blue', 'pixel_3_green', 'pixel_3_red', 'pixel_3_NIR',
       'pixel_4_blue', 'pixel_4_green', 'pixel_4_red', 'pixel_4_NIR',
       'pixel_5_blue', 'pixel_5_green', 'pixel_5_red', 'pixel_5_NIR',
       'pixel_6_blue', 'pixel_6_green', 'pixel_6_red', 'pixel_6_NIR',
       'pixel_7_blue', 'pixel_7_green', 'pixel_7_red', 'pixel_7_NIR',
       'pixel_8_blue', 'pixel_8_green', 'pixel_8_red', 'pixel_8_NIR',
       'pixel_9_blue', 'pixel_9_green', 'pixel_9_red', 'pixel_9_NIR',
       'pixel_10_blue', 'pixel_10_green', 'pixel_10_red', 'pixel_10_NIR',
       'pixel_11_blue', 'pixel_11_green', 'pixel_11_red', 'pixel_11_NIR',
       'pixel_12_blue', 'pixel_12_green', 'pixel_12_red', 'pixel_12_NIR',
       'pixel_13_blue', 'pixel_13_green', 'pixel_13_red', 'pixel_13_NIR',
       'pixel_14_blue', 'pixel_14_green', 'pixel_14_red', 'pixel_14_NIR',
       'pixel_15_blue', 'pixel_15_green', 'pixel_15_red', 'pixel_15_NIR',
       'pixel_16_blue', 'pixel_16_green', 'pixel_16_red', 'pixel_16_NIR',
       'pixel_17_blue', 'pixel_17_green', 'pixel_17_red', 'pixel_17_NIR',
       'pixel_18_blue', 'pixel_18_green', 'pixel_18_red', 'pixel_18_NIR',
       'pixel_19_blue', 'pixel_19_green', 'pixel_19_red', 'pixel_19_NIR',
       'pixel_20_blue', 'pixel_20_green', 'pixel_20_red', 'pixel_20_NIR']

# Apply the max() function to each list in the specified columns
for col in columns_to_fix:
    df_pixel_2022[col] = df_pixel_2022[col].apply(lambda x: max(x) if isinstance(x, list) else x)

df_pixel_2022

,name,pixel_1_blue,pixel_1_green,pixel_1_red,pixel_1_NIR,pixel_2_blue,pixel_2_green,pixel_2_red,pixel_2_NIR,pixel_3_blue,...,pixel_18_red,pixel_18_NIR,pixel_19_blue,pixel_19_green,pixel_19_red,pixel_19_NIR,pixel_20_blue,pixel_20_green,pixel_20_red,pixel_20_NIR
0,20220508_153525_43_24a5_3B_AnalyticMS_SR_clip_...,384.0,749.0,639.0,3910.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20220508_153525_43_24a5_3B_AnalyticMS_SR_clip_...,NaN,NaN,NaN,NaN,369.0,765.0,633.0,4157.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20220508_153525_43_24a5_3B_AnalyticMS_SR_clip_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,459.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20220508_153525_43_24a5_3B_AnalyticMS_SR_clip_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20220508_153525_43_24a5_3B_AnalyticMS_SR_clip_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
655,20221022_153215_03_2479_3B_AnalyticMS_SR_clip_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
656,20221022_153215_03_2479_3B_AnalyticMS_SR_clip_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
657,20221022_153215_03_2479_3B_AnalyticMS_SR_clip_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,784.0,3574.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
658,20221022_153215_03_2479_3B_AnalyticMS_SR_clip_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,469.0,819.0,1240.0,5020.0,NaN,NaN,NaN,NaN


In [35]:
# Remove the last '_pixel_X' from the 'name' column in agg_df

df_pixel_2022['name'] = df_pixel_2022['name'].str.rsplit('_pixel', n=1).str[0]



# Set the image name as the index for easier manipulation
df_pixel_2022 = df_pixel_2022.set_index('name')

# 1. Create a MultiIndex from your column names (e.g., 'pixel_3_blue' -> ['pixel_3', 'blue'])
df_pixel_2022.columns = pd.MultiIndex.from_tuples(
    [col.rsplit('_', 1) for col in  df_pixel_2022.columns],
    names=['pixel_id', 'band']
)

# 2. Stack the 'pixel_id' level. This pivots the pixel IDs from columns into rows,
#    automatically dropping all the NaN entries.
df_pixel_2022_tidy = df_pixel_2022.stack(level='pixel_id')

# 3. Reset the index to turn 'image_name' and 'pixel_id' from index levels into columns
df_pixel_2022_tidy = df_pixel_2022_tidy.reset_index()

# Optional: Clean up the 'pixel_id' column to be just a number
# df_tidy['tarp_id'] = df_tidy['tarp_id'].str.replace('tarp_', '').astype(int)

# Display the new, clean DataFrame
print(df_pixel_2022_tidy.head())

band                                           name pixel_id   blue  green  \
0     20220508_153525_43_24a5_3B_AnalyticMS_SR_clip  pixel_1  384.0  749.0   
1     20220508_153525_43_24a5_3B_AnalyticMS_SR_clip  pixel_2  369.0  765.0   
2     20220508_153525_43_24a5_3B_AnalyticMS_SR_clip  pixel_3  459.0  821.0   
3     20220508_153525_43_24a5_3B_AnalyticMS_SR_clip  pixel_4  384.0  809.0   
4     20220508_153525_43_24a5_3B_AnalyticMS_SR_clip  pixel_5  395.0  794.0   

band    red     NIR  
0     639.0  3910.0  
1     633.0  4157.0  
2     789.0  3761.0  
3     580.0  4467.0  
4     700.0  3750.0  


In [36]:
import pandas as pd

# Assuming your DataFrame is named 'df'
# If it's not loaded, you would load it first:
# df = pd.read_csv('your_data.csv') 

# 1. Pivot the table
# - index='name': makes each unique 'name' a new row
# - columns='tarp_id': uses the values from 'tarp_id' to create new columns
# - values=['blue', 'green', 'red', 'NIR']: the values to fill the new grid
df_pixel_2022_wide = df_pixel_2022_tidy.pivot_table(
    index='name', 
    columns='pixel_id', 
    values=['blue', 'green', 'red', 'NIR']
)

# 2. Flatten the multi-level columns
# The pivot creates hierarchical columns like ('blue', 'tarp_r1'). 
# This step joins them into a single name like 'tarp_r1_blue'.
df_pixel_2022_wide.columns = [f'{pixel}_{band}' for band, pixel in df_pixel_2022_wide.columns]

# 3. Reset the index to make 'name' a regular column again
df_pixel_2022_wide = df_pixel_2022_wide.reset_index()

# Display the new, wide DataFrame
print(df_pixel_2022_wide.head())

                                            name  pixel_1_NIR  pixel_10_NIR  \
0  20220508_153525_43_24a5_3B_AnalyticMS_SR_clip       3910.0        3853.0   
1  20220508_153527_74_24a5_3B_AnalyticMS_SR_clip       3882.0        3860.0   
2  20220509_154511_72_227a_3B_AnalyticMS_SR_clip       3895.0        3933.0   
3  20220510_150455_04_2465_3B_AnalyticMS_SR_clip       3645.0        3714.0   
4  20220510_154526_12_2424_3B_AnalyticMS_SR_clip       3893.0        3960.0   

   pixel_11_NIR  pixel_12_NIR  pixel_13_NIR  pixel_14_NIR  pixel_15_NIR  \
0        4100.0        3705.0        4150.0        5143.0        3828.0   
1        4068.0        3715.0        4094.0        5106.0        3821.0   
2        4022.0        3810.0        4212.0        4958.0        4063.0   
3        3895.0        3555.0        3942.0        4804.0        3788.0   
4        4109.0        3836.0        4205.0        5075.0        3988.0   

   pixel_16_NIR  pixel_17_NIR  ...  pixel_19_red  pixel_2_red  pixel_20_re

In [41]:
# 1. Create a dictionary to map old names to new names
rename_map = {}
for col in df_pixel_2022_wide.columns:
    if 'pixel_19' in col:
        # Replace 'pixel_19' with 'red_tarp'
        rename_map[col] = col.replace('pixel_19', 'red_tarp')
    elif 'pixel_20' in col:
        # Replace 'pixel_20' with 'white_tarp'
        rename_map[col] = col.replace('pixel_20', 'white_tarp')

# 2. Apply the renaming to the DataFrame
df_pixel_2022_wide = df_pixel_2022_wide.rename(columns=rename_map)

# Display the new column names to verify
print("New column names:")
print(df_pixel_2022_wide.columns)

New column names:
Index(['name', 'pixel_1_NIR', 'pixel_10_NIR', 'pixel_11_NIR', 'pixel_12_NIR',
       'pixel_13_NIR', 'pixel_14_NIR', 'pixel_15_NIR', 'pixel_16_NIR',
       'pixel_17_NIR', 'pixel_18_NIR', 'red_tarp_NIR', 'pixel_2_NIR',
       'white_tarp_NIR', 'pixel_3_NIR', 'pixel_4_NIR', 'pixel_5_NIR',
       'pixel_6_NIR', 'pixel_7_NIR', 'pixel_8_NIR', 'pixel_9_NIR',
       'pixel_1_blue', 'pixel_10_blue', 'pixel_11_blue', 'pixel_12_blue',
       'pixel_13_blue', 'pixel_14_blue', 'pixel_15_blue', 'pixel_16_blue',
       'pixel_17_blue', 'pixel_18_blue', 'red_tarp_blue', 'pixel_2_blue',
       'white_tarp_blue', 'pixel_3_blue', 'pixel_4_blue', 'pixel_5_blue',
       'pixel_6_blue', 'pixel_7_blue', 'pixel_8_blue', 'pixel_9_blue',
       'pixel_1_green', 'pixel_10_green', 'pixel_11_green', 'pixel_12_green',
       'pixel_13_green', 'pixel_14_green', 'pixel_15_green', 'pixel_16_green',
       'pixel_17_green', 'pixel_18_green', 'red_tarp_green', 'pixel_2_green',
       'white_tarp_gree

In [42]:
df_pixel_2022_wide.to_csv(r'D:\Capstone\Data\2022\pixel_2022_wide.csv', index=False)

#  New tryyyyy